# 🚀 Dərs 3: Qabaqcıl RAG Texnikaları (Advanced RAG: Semantic Chunking, Query Rewriting & Prompt Compression)

Əvvəlki dərsdə **LangChain** və **Ragas** vasitəsilə sadə (*Naive*) RAG sistemimizi qurduq və qiymətləndirdik. Orada ssenarini **tək-tək sətirlər** şəklində saxladığımız üçün bəzi suallarda ətraf kontekst itmişdi.

Bu dərsdə həmin sistemi **3 qabaqcıl RAG (Advanced RAG)** texnikası ilə təkmilləşdirəcəyik:
1. **Semantik Parçalama (`Semantic Chunking` — Pre-Retrieval):** Mətni kor-koranə sabit uzunluqlu hissələrə və ya tək sətirlərə bölmək əvəzinə, cümlələrin **embedding vektorları arasındakı məna fərqinə** görə böləcəyik ki, eyni mövzudan bəhs edən ardıcıl replikalar bir parçada (*chunk*) birləşsin.
2. **Sorğunun Yenidən Yazılması (`Query Rewriting / RePhraseQueryRetriever` — Pre-Retrieval):** İstifadəçinin verdiyi sualı vektor bazasında axtarışa göndərməzdən əvvəl LLM vasitəsilə təmizləyib axtarış üçün ən optimal açar ifadəyə çevirəcəyik.
3. **Kontekstin Sıxılması və Filtrlənməsi (`Prompt Compression / EmbeddingsFilter` — Post-Retrieval):** Vektor bazasından qayıdan 10 nəticənin hamısını kor-koranə LLM-ə göndərmək əvəzinə, sualla semantik oxşarlığı müəyyən həddən (`similarity_threshold`) aşağı olan əlaqəsiz parçaları süzgəcdən keçirib atacağıq.

İlk olaraq əvvəlki dərslərdə olduğu kimi `./sample_data/tng` qovluğundakı ssenarilərdən **DATA** obrazının replikalarını çıxaraq:

In [ ]:
import os
import re
import random
import sys

# Lokal mühitdə köhnə 'google-colab' paketinin digər kitabxanalarda xəta verməsinin qarşısını alırıq:
sys.modules["google.colab"] = None

# Bütün çıxarılmış DATA replikalarının saxlanacağı əsas siyahı
dialogues = []

def strip_parentheses(s):
    """Mötərizə daxilindəki səhnə qeydlərini mətndən silir: məs. '(hesitates)'"""
    return re.sub(r'\(.*?\)', '', s)

def is_single_word_all_caps(s):
    """Sətrin tək sözdən ibarət və BÖYÜK HƏRFLƏRLƏ yazılmış obraz adı olub-olmadığını yoxlayır"""
    words = s.split()
    if len(words) != 1:
        return False
    if bool(re.search(r'\d', words[0])):
        return False
    return words[0].isupper()

def extract_character_lines(file_path, character_name):
    """Verilmiş ssenari faylından konkret obrazın (DATA) bütün replikalarını çıxarır"""
    lines = []
    with open(file_path, 'r') as script_file:
        try:
            lines = script_file.readlines()
        except UnicodeDecodeError:
            pass

    is_character_line = False
    current_line = ''
    current_character = ''
    for line in lines:
        strippedLine = line.strip()
        if is_single_word_all_caps(strippedLine):
            is_character_line = True
            current_character = strippedLine
        elif (line.strip() == '') and is_character_line:
            is_character_line = False
            dialog_line = strip_parentheses(current_line).strip()
            dialog_line = dialog_line.replace('"', "'")
            if current_character == character_name and len(dialog_line) > 0:
                dialogues.append(dialog_line)
            current_line = ''
        elif is_character_line:
            current_line += line.strip() + ' '

def process_directory(directory_path, character_name):
    """Qovluqdakı bütün ssenari fayllarını ardıcıl emal edir"""
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):
            extract_character_lines(file_path, character_name)


In [ ]:
# './sample_data/tng' qovluğundakı bütün ssenarilərdən 'DATA' obrazının replikalarını çıxarırıq
process_directory("./sample_data/tng", 'DATA')

### ✅ Məlumatların Yoxlanılması (Sanity Check)

Replikaların düzgün oxunduğundan əmin olmaq üçün ilk sətri (`dialogues[0]`) və ümumi sayını ekrana çap edək:

In [ ]:
print(f"Ümumi replika sayı: {len(dialogues)}")
print("İlk replika:", dialogues[0])

### 📦 Lazımi Kitabxanaların Quraşdırılması (`google-genai`, `ragas`, `langchain`, `langchain_experimental`)

RAG modelimiz üçün rəsmi **Google GenAI SDK (`google-genai`)**, qiymətləndirmə üçün **Ragas** və **LangChain** paketlərini quraşdıraq:

In [ ]:
# Google GenAI SDK-nı yeniləyirik
!pip install google-genai python-dotenv --upgrade

Nəticələri ölçmək üçün **Ragas** və **LangChain** paketlərini quraşdıraq:

In [ ]:
# Ragas + LangChain paketlərini quraşdırırıq
!pip install -U ragas langchain langchain-core langchain-community datasets nest_asyncio

### 🔑 Addım 1: Google Gemini API (`from google.genai import types`) və LangChain Adapterlərinin Qurulması

Burada rəsmi **`google-genai`** kitabxanasından (`from google import genai` və `from google.genai import types`) istifadə edərək:
1. API açarını mühit dəyişənindən (`GEMINI_API_KEY` / `GOOGLE_API_KEY`), `.env` faylından və ya birbaşa daxiletmədən (`getpass`) oxuyuruq.
2. **`GeminiEmbeddings`** (`gemini-embedding-001`) və **`ChatGemini`** (`gemini-2.5-flash`) adapter siniflərini yaradırıq ki, həm **`SemanticChunker`**, həm **`RePhraseQueryRetriever`**, həm **`EmbeddingsFilter`**, həm də **Ragas `evaluate()`** birbaşa `types.EmbedContentConfig` və `types.GenerateContentConfig` ilə işləsin.

In [ ]:
import os
import time
import asyncio
from typing import List, Optional, Any
from getpass import getpass

from google import genai
from google.genai import types
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration

# Əgər layihədə .env faylı varsa, açarları avtomatik yükləyirik
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 1. Gemini API açarını mühitdən oxuyuruq və ya istifadəçidən istəyirik
api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
if not api_key:
    api_key = getpass('Gemini API açarınızı daxil edin (AIzaSy...): ').strip().strip('"').strip("'")

# 2. Rəsmi Google GenAI Client yaradırıq
client = genai.Client(api_key=api_key)

# 3. Google GenAI SDK (`types.EmbedContentConfig`) əsasında LangChain Embeddings sinfi
class GeminiEmbeddings(Embeddings):
    """google-genai və types.EmbedContentConfig istifadə edən LangChain Embeddings adapteri"""
    def __init__(self, model: str = "gemini-embedding-001", dimensions: int = 128):
        self.model = model
        self.dimensions = dimensions

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        batch_size = 100  # Gemini API bir sorğuda maksimum 100 mətn qəbul edir
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else " " for t in texts[i:i + batch_size]]
            while True:
                try:
                    response = client.models.embed_content(
                        model=self.model,
                        contents=batch,
                        config=types.EmbedContentConfig(
                            output_dimensionality=self.dimensions
                        )
                    )
                    all_embeddings.extend([e.values for e in response.embeddings])
                    break
                except Exception as e:
                    if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                        print("Embedding API sorğu limiti (429), 20 saniyə gözlənilir...")
                        time.sleep(20)
                    else:
                        raise e
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return await asyncio.to_thread(self.embed_documents, texts)

    async def aembed_query(self, text: str) -> List[float]:
        return await asyncio.to_thread(self.embed_query, text)


# 4. Google GenAI SDK (`types.GenerateContentConfig`) əsasında LangChain ChatModel sinfi
class ChatGemini(BaseChatModel):
    """google-genai və types.GenerateContentConfig istifadə edən LangChain Chat adapteri"""
    model_name: str = "gemini-2.5-flash"
    temperature: float = 0.0

    @property
    def _llm_type(self) -> str:
        return "google_genai_chat"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        sys_parts = []
        user_parts = []
        for m in messages:
            if isinstance(m, SystemMessage) or getattr(m, 'type', '') == 'system':
                sys_parts.append(str(m.content))
            else:
                user_parts.append(str(m.content))

        sys_instruction = "\n".join(sys_parts) if sys_parts else None
        prompt_text = "\n\n".join(user_parts) if user_parts else " "

        while True:
            try:
                response = client.models.generate_content(
                    model=self.model_name,
                    contents=prompt_text,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_instruction,
                        temperature=self.temperature
                    )
                )
                text = response.text if response.text else ""
                return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])
            except Exception as e:
                if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                    print("Chat API sorğu limiti (429), 15 saniyə gözlənilir...")
                    time.sleep(15)
                else:
                    raise e

    async def _agenerate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        return await asyncio.to_thread(self._generate, messages, stop=stop, run_manager=run_manager, **kwargs)

print("Google Gemini Client və LangChain adapterləri uğurla hazırlandı!")

### 📝 Addım 2: Replikaların Mətn Faylına Yazılması (`data_lines.txt`)

Semantik parçalama (`SemanticChunker`) ardıcıl gələn replikaları məna yaxınlığına görə birləşdirdiyi üçün, **"Lal"** haqqındakı replikaları onların **ətrafındakı qonşu replikalarla birlikdə (ardıcıl blok şəklində)** `./sample_data/data_lines.txt` faylına yazırıq.

> 💡 **Niyə ətraf replikalar vacibdir?**
> Məsələn, Data bir replikada *"Computer, Lal -- gender sequence finalists... begin..."* deyir, növbəti replikalarda isə *"An Andorian female... A human male... The human female. A Klingon male."* sadalayır. Həmin sonrakı sətirlərdə *"Lal"* sözü birbaşa keçməsə də, ardıcıl gəldikləri üçün `SemanticChunker` onları eyni semantik parçada birləşdirəcək!

In [ ]:
# Pulsuz API kvotasında (dəqiqədə 100 embedding sorğusu) sürətli işləməsi üçün
# 'Lal' adı keçən replikaları və onların ətrafındakı (+/- 2) ardıcıl kontekst sətirlərini seçirik.
# (Ödənişli planda bütün sətirlər üçün MAX_LINES = len(dialogues) edə bilərsiniz)
MAX_LINES = 80

if MAX_LINES < len(dialogues):
    selected_indices = set()
    for idx, line in enumerate(dialogues):
        if 'Lal' in line:
            # Ətraf konteksti qorumaq üçün hər 'Lal' sətrinin ətrafındakı qonşu replikaları da daxil edirik
            for neighbor in range(max(0, idx - 1), min(len(dialogues), idx + 3)):
                selected_indices.add(neighbor)
    ordered_indices = sorted(selected_indices)[:MAX_LINES]
    selected_dialogues = [dialogues[i] for i in ordered_indices]
else:
    selected_dialogues = dialogues

# Seçilmiş replikaları tək bir mətn faylına yazırıq:
with open("./sample_data/data_lines.txt", "w+", encoding="utf-8") as f:
    for line in selected_dialogues:
        f.write(line + "\n")

print(f"'./sample_data/data_lines.txt' faylına {len(selected_dialogues)} ardıcıl replika yazıldı.")

### 🧪 Addım 3: `langchain_experimental` Paketinin Quraşdırılması

İndi isə ən maraqlı hissəyə keçirik! Semantik parçalama (`SemanticChunker`) kimi müasir eksperimental RAG alətlərindən istifadə etmək üçün **`langchain_experimental`** və **`faiss-cpu`** paketlərini quraşdıraq:

In [ ]:
# Eksperimental LangChain alətlərini (SemanticChunker) və FAISS-i quraşdırırıq
!pip install langchain_experimental faiss-cpu

### 🧩 Addım 4: Semantik Parçalama (`Semantic Chunking` — Pre-Retrieval)

Əvvəlki dərsdə vektor bazasında hər replikanı tək bir sətir kimi saxlamışdıq. Standart `LangChain` yanaşmasında isə mətn sabit uzunluqlu bloklara (*fixed-length chunking*) bölünür (aşağıda şərhə alınmış kod məhz bunu edir).

Biz isə daha ağıllı üsul olan **Semantik Parçalama (`SemanticChunker`)** tətbiq edəcəyik:
- **Necə işləyir?** Ardıcıl sətirlərin/cümlələrin embedding vektorları hesablanır. Əgər iki qonşu hissə arasındakı kosinus məsafəsi müəyyən həddi (*breakpoint threshold*) keçirsə, deməli mövzu dəyişib və yeni parça (*chunk*) başladılır.
- **Hədd üsulları (`breakpoint_threshold_type`):** Biz burada **`"percentile"` (faiz dərəcəsi)** üsulundan istifadə edirik, lakin siz **`"standard_deviation"` (standart meyl)** və ya **`"interquartile"` (kvartillərarası diapazon)** üsullarını da sınaqdan keçirə bilərsiniz. RAG sistemini tənzimlənən hiperparametrləri olan böyük bir maşın öyrənməsi modeli kimi düşünə bilərsiniz!

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.vectorstores import FAISS

# Həm yeni (langchain_classic), həm də standart langchain versiyaları ilə tam uyğunluq:
try:
    from langchain_classic.indexes import VectorstoreIndexCreator
except ImportError:
    from langchain.indexes import VectorstoreIndexCreator

# Google Gemini Embedding modelimizi (gemini-embedding-001, 128 ölçü) yaradırıq:
embeddings = GeminiEmbeddings(model="gemini-embedding-001", dimensions=128)

# -------------------------------------------------------------------------
# Sadə sabit ölçülü parçalama (Fixed-size chunking — müqayisə üçün şərhə alınıb):
# loader = TextLoader("./sample_data/data_lines.txt", encoding="utf-8")
# index = VectorstoreIndexCreator(embedding=embeddings).from_loaders([loader])
# -------------------------------------------------------------------------

# Əvəzində Semantik Parçalama (SemanticChunker) istifadə edirik:
# sentence_split_regex=r"\n+" vasitəsilə hər dialoq sətrini bütöv vahid kimi götürüb
# mənaca yaxın olan ardıcıl replikaları eyni sənəd blokunda (chunk) birləşdiririk:
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    sentence_split_regex=r"\n+"
)

with open("./sample_data/data_lines.txt", encoding="utf-8") as f:
    data_lines = f.read()

# Mətni semantik cəhətdən mənalı bloklara (docs) bölürük
docs = text_splitter.create_documents([data_lines])
print(f"Semantik parçalama nəticəsində {len(docs)} ədəd mənalı sənəd bloku (chunk) yaradıldı.")

# Yaradılmış semantik blokları vektor bazasına (index) yükləyirik
index = VectorstoreIndexCreator(
    embedding=embeddings,
    vectorstore_cls=FAISS
).from_documents(docs)

### ⚙️ Addım 5: Baza RAG Komponentlərinin və `k=10` Axtarış Sisteminin Qurulması

İndi RAG boru kəmərimizi quraq:
- Modelə yenə **Sistem Təlimatı (`system_prompt`)** vasitəsilə **Leytenant Komandir Data** olduğunu bildiririk.
- Vektor bazasından ilkin olaraq **10 ədəd (`k=10`)** ən yaxın kontekst blokunu gətirən `retriever` yaradırıq (`k` özü də tənzimlənə bilən mühüm hiperparametrdir).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Həm langchain_classic, həm də standart langchain paketləri ilə uyğun import:
try:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.retrievers import RePhraseQueryRetriever, ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import EmbeddingsFilter
except ImportError:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.retrievers import RePhraseQueryRetriever, ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import EmbeddingsFilter

# 1. Google Gemini Chat modelini (temperature=0) yaradırıq
llm = ChatGemini(model_name="gemini-2.5-flash", temperature=0.0)

# 2. Sistem təlimatı (System Prompt)
system_prompt = (
    "You are Lt. Commander Data from Star Trek: The Next Generation. "
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# 3. Vektor bazasından ilkin olaraq Top-10 konteksti gətirən baza Retriever
retriever = index.vectorstore.as_retriever(search_kwargs={'k': 10})

### ✍️ Addım 6: Sorğunun Yenidən Yazılması (`Query Rewriting / RePhraseQueryRetriever` — Pre-Retrieval)

İstifadəçilər bəzən sualları danışıq dilində, artıq sözlərlə və ya qarışıq formada verirlər. **`RePhraseQueryRetriever`** bizim standart `retriever` obyektini bükərək:
1. İstifadəçinin sualını əvvəlcə LLM-ə (`ChatGemini`) göndərir.
2. LLM sualdakı lazımsız sözləri ataraq vektor bazasında axtarış üçün **təmizlənmiş və fokuslanmış sorğu** yaradır.
3. Vektor bazasında axtarışı həmin yenidən yazılmış sorğu ilə aparır.

In [ ]:
# Baza retriever-in üzərinə LLM əsaslı Sorğu Yenidən Yazan (RePhraseQueryRetriever) əlavə edirik:
retriever_from_llm = RePhraseQueryRetriever.from_llm(
    retriever=retriever, llm=llm
)

### 🗜️ Addım 7: Kontekstin Sıxılması və Filtrlənməsi (`Prompt Compression / EmbeddingsFilter` — Post-Retrieval)

İndi isə **Post-Retrieval (axtarış sonrası)** mərhələsində **Prompt Compression** əlavə edirik.

Bəzi üsullar tapılmış kontekstləri sıxmaq üçün yenidən LLM-ə müraciət edir, lakin bu həm vaxt aparır, həm də əlavə xərc yaradır. Daha sürətli və effektiv üsul **`EmbeddingsFilter`** istifadə etməkdir:
- Vektor bazasından gətirilmiş 10 kontekstin hər birinin sualla **embedding oxşarlığı (cosine similarity)** yoxlanılır.
- Oxşarlığı müəyyən etdiyimiz həddən (`similarity_threshold`) aşağı olan əlaqəsiz kontekstlər atılır.

> 📌 **Hiperparametr Qeydi (`similarity_threshold`):**
> OpenAI-nin `text-embedding-ada-002` modelində bütün mətnlər dar bir bucaqda yerləşdiyi üçün hədd adətən `0.76` götürülür. **Google Gemini (`gemini-embedding-001`)** modelində isə kosinus oxşarlığı daha geniş şkalada paylandığı üçün uyğun sənədlər adətən **`0.55 – 0.80`**, əlaqəsiz sənədlər isə **`< 0.55`** aralığında olur. Buna görə `similarity_threshold=0.55` təyin edirik.

In [ ]:
# 1. Suala kifayət qədər oxşar olmayan kontekstləri ələyən EmbeddingsFilter yaradırıq
# (Gemini embedding-001 üçün optimal hədd: 0.55; OpenAI ada-002 üçün: 0.76)
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.55)

# 2. Yenidən yazılmış sorğu axtarışını (retriever_from_llm) sıxılma filtri ilə birləşdiririk
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, base_retriever=retriever_from_llm
)

# 3. Yekun sual-cavab zəncirini (chain) qururuq
question_answer_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(compression_retriever, question_answer_chain)

### 🔍 Sıxılma Filtrinin (`compression_retriever`) Yoxlanılması

`compression_retriever`-in həqiqətən işlədiyini görmək üçün onu birbaşa bir sual üzərində sınayaq. Vektor bazasından 10 nəticə (`k=10`) istəməyimizə baxmayaraq, yalnız oxşarlıq həddini (`similarity_threshold`) keçən həqiqətən uyğun sənədlərin saxlanıldığını görəcəyik:

In [ ]:
def pretty_print_docs(docs):
    print(f"Filtrləmədən keçən sənəd sayı: {len(docs)} / 10\n")
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

compressed_docs = compression_retriever.invoke("How many calculations per second can Lal complete?")
pretty_print_docs(compressed_docs)

### 🧪 Addım 8: Tam Qabaqcıl RAG Zəncirinin (`chain`) və Sorğu Yenidən Yazma Loqunun Sınağı

İndi `logging` modulunu aktivləşdirək ki, `RePhraseQueryRetriever`-in sualımızı arxa planda necə yenidən yazdığını (`INFO:...:Re-phrased question: ...`) əyani görək, ardınca həm tapılmış semantik blokları (`SOURCE DOCUMENTS`), həm də yekun cavabı (`RESULT`) çap edək:

In [ ]:
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.re_phraser").setLevel(logging.INFO)
logging.getLogger("langchain_classic.retrievers.re_phraser").setLevel(logging.INFO)

question = "Tell me about your daughter, Lal."

result = chain.invoke({"input": question})
print("SOURCE DOCUMENTS:\n")
for doc in result["context"]:
    print(doc)
print("\nRESULT:\n")
print(result["answer"])

### 📏 Addım 9: Ragas Qiymətləndirməsi Üçün Test Sualları və Etalon Cavablar (`Ground Truth`)

Əvvəlki dərsdəki **eyni 6 test sualı (`eval_questions`)** və **etalon cavablar (`eval_answers`)** üzərində yeni Qabaqcıl RAG modelimizi yoxlayaq ki, nəticələri birbaşa müqayisə edə bilək:

In [ ]:
# Sınaq sualları (Test Questions)
eval_questions = [
    "Is Lal your daughter?",
    "How many calculations per second can Lal complete?",
    "Does Lal have emotions?",
    "What goal did you have for Lal?",
    "How was Lal's species and gender chosen?",
    "What happened to Lal?"
]

# Hər bir sual üçün həqiqi etalon cavablar (Ground Truth Answers)
eval_answers = [
    "Yes, Lal is my daughter. I created Lal.",
    "Lal is capable of completing sixty trillion calculations per second.",
    "Yes, unlike myself, Lal proved able to feel emotions such as fear and love.",
    "My goal for Lal was for her to enter Starfleet Academy.",
    "Lal chose her own identity as a human female, after consulting with Counselor Troi.",
    "Lal experienced a cascade failure in her neural net, triggered by distress from her impending separation from me to Galor IV. I deactivated Lal once she suffered complete neural system failure."
]


### 🔍 Cavab Strukturunun Yoxlanılması

`chain.invoke()` metodunun qaytardığı lüğət (`dict`) strukturunu (`input`, `context`, `answer`) görmək üçün suallardan birini sınaqdan keçirək:

In [ ]:
result = chain.invoke({"input": eval_questions[1]})
print("Sual:", result["input"])
print("Tapılmış kontekst sayı:", len(result["context"]))
print("Cavab:", result["answer"])

### 📥 Addım 10: Bütün Test Sualları Üzrə Cavabların (`answers`) və Kontekstlərin (`contexts`) Toplanması

Hər 6 sual üçün modelin cavabını və `compression_retriever` tərəfindən seçilmiş kontekst mətnlərini toplayırıq:

In [ ]:
answers = []
contexts = []

for question in eval_questions:
    response = chain.invoke({"input": question})
    answers.append(response["answer"])
    contexts.append([context.page_content for context in response["context"]])

print(f"{len(answers)} cavab və {len(contexts)} kontekst siyahısı toplandı.")

### 🗂️ Addım 11: Məlumatların Hugging Face `Dataset` Formatına Çevrilməsi

Toplanmış nəticələri **Ragas** üçün tələb olunan Hugging Face `Dataset` formatına çeviririk:

In [ ]:
from datasets import Dataset

response_dataset = Dataset.from_dict({
    "question": eval_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": eval_answers
})

# İlk sualın nəticə və kontekst strukturuna baxaq:
response_dataset[0]

### ⚖️ Addım 12: Ragas Metrikalarının və Hakim Modelin (`Evaluator LLM`) Təyin Edilməsi

Əvvəlki dərsdəki eyni 5 metrikanı (`faithfulness`, `answer_relevancy`, `context_recall`, `context_precision`, `answer_correctness`) və Google Gemini əsaslı hakim modelimizi (`evaluator_llm`, `evaluator_embeddings`) hazırlayırıq:

In [ ]:
from ragas import evaluate, RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_recall,
    context_precision,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    answer_correctness,
]

# Google Gemini LLM və Embeddings obyektlərimizi Ragas üçün bükürük
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

### 🚀 Addım 13: `evaluate()` Funksiyasının İşə Salınması

İndi Qabaqcıl RAG sistemimizi Ragas ilə qiymətləndirək:

In [ ]:
import nest_asyncio
nest_asyncio.apply()

results = evaluate(
    response_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(max_workers=2, timeout=180)
)

### 📈 Addım 14: Qiymətləndirmə Nəticələrinin Müqayisəsi

Gəlin nəticələrə baxaq! Əvvəlki sadə (*Naive RAG*) yanaşma ilə müqayisədə:
- **Semantik Parçalama (`SemanticChunker`)** sayəsində bir-biri ilə bağlı olan ardıcıl replikalar (məsələn, Lal-ın növ və cins seçimi səhnəsi) bütöv bir kontekst kimi tapılır.
- **`RePhraseQueryRetriever`** və **`EmbeddingsFilter`** sayəsində həm sualın ifadə dəqiqliyi (`answer_relevancy`), həm də kontekst dəqiqliyi (`context_precision` və `context_recall`) əhəmiyyətli dərəcədə yüksəlir!

In [ ]:
# Bütün suallar üzrə orta metrika nəticələri
results

In [ ]:
# Hər bir sual üzrə detallı qiymətləndirmə cədvəli
results.to_pandas()

### 💡 Yekun Tövsiyələr və Hiperparametrlərin Tənzimlənməsi

Yuxarıda təsvir olunan **hiperparametrlərlə** (`k`, `breakpoint_threshold_type`, `similarity_threshold`) təcrübə apararaq nəticələri daha da yaxşılaşdırmağa çalışın! Unutmayın ki, rəqəmsal metrikalar hər şeyi demir — cavabların keyfiyyəti barədə sizin subyektiv qiymətləndirməniz də ən azı metrikalar qədər önəmlidir.

Bu cür Qabaqcıl RAG sistemini real istehsalat (*production*) mühitində işlətmək üçün onu Jupyter kitabçasından müstəqil Python skriptinə çevirib **[LangServe](https://www.langchain.com/langserve)** və ya **FastAPI** vasitəsilə REST API xidməti kimi təqdim edə bilərsiniz.